# Small AutoGraph Evaluation Sandbox

This notebook builds a small batch of 20 synthetic graphs with about 100 nodes each, stores them as `SimpleGraphData`, treats half as training graphs and half as generated graphs, tokenizes the generated half, and evaluates them with the repo's `Evaluation` logic.

In [7]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path.cwd().parent))

from evaluation.Evaluator import Evaluator
from graph_tokenization import (
    AutoGraphTokenizer,
    SimpleGraphData,
)

In [8]:
def erdos_renyi_simple_graph_data(
    num_nodes: int = 100,
    edge_prob: float = 0.035,
    seed: int | None = None,
) -> SimpleGraphData:
    """Create an undirected random graph represented as SimpleGraphData."""
    generator = torch.Generator()
    if seed is not None:
        generator.manual_seed(seed)

    adjacency = torch.rand((num_nodes, num_nodes), generator=generator) < edge_prob
    adjacency = torch.triu(adjacency, diagonal=1)
    src, dst = adjacency.nonzero(as_tuple=True)

    edge_index = torch.cat(
        [torch.stack([src, dst]), torch.stack([dst, src])],
        dim=1,
    ).long()

    return SimpleGraphData(edge_index=edge_index, num_nodes=num_nodes)

In [9]:
num_graphs = 20
num_nodes = 1000

graphs = [
    erdos_renyi_simple_graph_data(num_nodes=num_nodes, edge_prob=0.035, seed=10_000 + i)
    for i in range(num_graphs)
]

train_data = graphs[:10]
generated_data = graphs[10:]

len(train_data), len(generated_data), generated_data[0].edge_index.shape

(10, 10, torch.Size([2, 34752]))

In [10]:
tokenizer = AutoGraphTokenizer(max_length=-1, undirected=True, rng=123)
tokenizer.set_num_nodes(num_nodes)

generated_tokens = [tokenizer.tokenize(graph) for graph in generated_data]
max_token_length = max(tokens.numel() for tokens in generated_tokens)

tokenized_generated_batch = torch.full(
    (len(generated_tokens), max_token_length),
    fill_value=tokenizer.pad,
    dtype=torch.long,
)
for row, tokens in enumerate(generated_tokens):
    tokenized_generated_batch[row, : tokens.numel()] = tokens

tokenized_generated_batch.shape

torch.Size([10, 19621])

In [11]:
evaluator = Evaluator()
metrics = evaluator(tokenized_generated_batch, train_data)
metrics

{'validity': 1.0,
 'uniqueness': 1.0,
 'novelty': 1.0,
 'max_degree': 54.5,
 'min_degree': 18.4,
 'avg_degree': 34.9336,
 'avg_num_connected_components': 1.0,
 'avg_largest_connected_component_size': 1000.0,
 'avg_nodes': 1000.0,
 'min_nodes': 1000,
 'max_nodes': 1000,
 'avg_edges': 17466.8,
 'min_edges': 17334,
 'max_edges': 17622,
 'ple_mean': np.float64(1.2362828945362652),
 'ple_std': np.float64(0.00035572276071431906),
 'edge_overlap_mean': 0.03883977151005838,
 'edge_overlap_std': 0.0007282349031487238,
 'edge_overlap_num_graphs': 10}

In [12]:
decoded_example = tokenizer.decode(tokenized_generated_batch[0])

{
    "num_train_graphs": len(train_data),
    "num_generated_graphs": len(generated_data),
    "token_batch_shape": tuple(tokenized_generated_batch.shape),
    "decoded_example_nodes": decoded_example.num_nodes,
    "decoded_example_edges": decoded_example.edge_index.shape[1],
    "metrics": metrics,
}

{'num_train_graphs': 10,
 'num_generated_graphs': 10,
 'token_batch_shape': (10, 19621),
 'decoded_example_nodes': 1000,
 'decoded_example_edges': 34752,
 'metrics': {'validity': 1.0,
  'uniqueness': 1.0,
  'novelty': 1.0,
  'max_degree': 54.5,
  'min_degree': 18.4,
  'avg_degree': 34.9336,
  'avg_num_connected_components': 1.0,
  'avg_largest_connected_component_size': 1000.0,
  'avg_nodes': 1000.0,
  'min_nodes': 1000,
  'max_nodes': 1000,
  'avg_edges': 17466.8,
  'min_edges': 17334,
  'max_edges': 17622,
  'ple_mean': np.float64(1.2362828945362652),
  'ple_std': np.float64(0.00035572276071431906),
  'edge_overlap_mean': 0.03883977151005838,
  'edge_overlap_std': 0.0007282349031487238,
  'edge_overlap_num_graphs': 10}}